In [14]:
from openai import OpenAI
from typing import List, Dict, Any, Optional

In [15]:
class UploadClient:
    """Client for managing large file uploads in parts.
    
    This client provides methods for:
    - Creating uploads (for files up to 8 GB)
    - Adding parts to uploads
    - Completing uploads
    - Canceling uploads
    
    Uploads allow you to upload large files in multiple parts, with each part
    being at most 64 MB. Parts can be uploaded in parallel for better performance.
    """
    
    def __init__(self, api_key: Optional[str] = None):
        """Initialize the UploadClient.
        
        Args:
            api_key: Optional OpenAI API key. If not provided, will use OPENAI_API_KEY env variable.
        """
        self.client = OpenAI(api_key=api_key) if api_key else OpenAI()
    
    def create(
        self,
        filename: str,
        purpose: str,
        bytes: int,
        mime_type: str,
        expires_after: Optional[Dict[str, Any]] = None
    ) -> Dict[str, Any]:
        """Create an intermediate Upload object that you can add Parts to.
        
        Currently, an Upload can accept at most 8 GB in total and expires after an hour.
        
        Args:
            filename: The name of the file to upload
            purpose: The intended purpose of the uploaded file (e.g., 'fine-tune', 'assistants')
            bytes: The number of bytes in the file you are uploading
            mime_type: The MIME type of the file (must match supported types for your purpose)
            expires_after: The expiration policy for the upload
        
        Returns:
            The Upload object with status pending
        """
        kwargs = {
            'filename': filename,
            'purpose': purpose,
            'bytes': bytes,
            'mime_type': mime_type
        }
        if expires_after is not None:
            kwargs['expires_after'] = expires_after
        
        upload = self.client.uploads.create(**kwargs)
        return upload
    
    def add_part(
        self,
        upload_id: str,
        data
    ) -> Dict[str, Any]:
        """Add a Part to an Upload object.
        
        Each Part can be at most 64 MB, and you can add Parts until you hit the Upload 
        maximum of 8 GB. Multiple Parts can be added in parallel.
        
        Args:
            upload_id: The ID of the Upload
            data: The chunk of bytes for this Part (file-like object opened in binary mode)
        
        Returns:
            The upload Part object
        """
        part = self.client.uploads.parts.create(
            upload_id=upload_id,
            data=data
        )
        return part
    
    def complete(
        self,
        upload_id: str,
        part_ids: List[str],
        md5: Optional[str] = None
    ) -> Dict[str, Any]:
        """Complete the Upload.
        
        The returned Upload object contains a nested File object that is ready to use.
        The number of bytes uploaded must match the number initially specified.
        
        Args:
            upload_id: The ID of the Upload
            part_ids: The ordered list of Part IDs
            md5: Optional md5 checksum for the file contents to verify upload integrity
        
        Returns:
            The Upload object with status completed and a file property containing the created File
        """
        kwargs = {
            'upload_id': upload_id,
            'part_ids': part_ids
        }
        if md5 is not None:
            kwargs['md5'] = md5
        
        completed_upload = self.client.uploads.complete(**kwargs)
        return completed_upload
    
    def cancel(self, upload_id: str) -> Dict[str, Any]:
        """Cancel the Upload.
        
        No Parts may be added after an Upload is cancelled.
        
        Args:
            upload_id: The ID of the Upload
        
        Returns:
            The Upload object with status cancelled
        """
        cancelled_upload = self.client.uploads.cancel(upload_id=upload_id)
        return cancelled_upload

In [16]:
from typing import Optional


class FileClient:
    """Client for managing OpenAI Files.
    
    This client provides methods for:
    - Uploading files
    - Listing files
    - Retrieving file metadata
    - Deleting files
    - Retrieving file content
    """
    
    def __init__(self, api_key: Optional[str] = None):
        """Initialize the FileClient.
        
        Args:
            api_key: Optional OpenAI API key. If not provided, will use OPENAI_API_KEY env variable.
        """
        self.client = OpenAI(api_key=api_key) if api_key else OpenAI()
    
    def create(
        self,
        file,
        purpose: str,
        expires_after: Optional[Dict[str, Any]] = None
    ) -> Dict[str, Any]:
        """Upload a file that can be used across various endpoints.
        
        Individual files can be up to 512 MB, and the size of all files uploaded by one 
        organization can be up to 1 TB.
        
        Args:
            file: The File object (not file name) to be uploaded (e.g., open("mydata.jsonl", "rb"))
            purpose: The intended purpose of the uploaded file. One of:
                - assistants: Used in the Assistants API
                - batch: Used in the Batch API
                - fine-tune: Used for fine-tuning
                - vision: Images used for vision fine-tuning
                - user_data: Flexible file type for any purpose
                - evals: Used for eval data sets
            expires_after: The expiration policy for a file (default: batch files expire after 30 days)
        
        Returns:
            The uploaded File object
        """
        kwargs = {
            'file': file,
            'purpose': purpose
        }
        if expires_after is not None:
            kwargs['expires_after'] = expires_after
        
        uploaded_file = self.client.files.create(**kwargs)
        return uploaded_file
    
    def list(
        self,
        purpose: Optional[str] = None,
        limit: int = 10000,
        order: str = "desc",
        after: Optional[str] = None
    ) -> Dict[str, Any]:
        """List files.
        
        Args:
            purpose: Only return files with the given purpose
            limit: Number of objects to return (1-10000, default 10000)
            order: Sort order by created_at timestamp ('asc' or 'desc', default 'desc')
            after: Cursor for pagination (object ID to start after)
        
        Returns:
            A list of File objects
        """
        kwargs = {'limit': limit, 'order': order}
        if purpose is not None:
            kwargs['purpose'] = purpose
        if after is not None:
            kwargs['after'] = after
        
        files = self.client.files.list(**kwargs)
        return files
    
    def retrieve(self, file_id: str) -> Dict[str, Any]:
        """Retrieve information about a specific file.
        
        Args:
            file_id: The ID of the file to retrieve
        
        Returns:
            The File object matching the specified ID
        """
        file = self.client.files.retrieve(file_id)
        return file
    
    def delete(self, file_id: str) -> Dict[str, Any]:
        """Delete a file and remove it from all vector stores.
        
        Args:
            file_id: The ID of the file to delete
        
        Returns:
            Deletion status
        """
        deleted_file = self.client.files.delete(file_id)
        return deleted_file
    
    def retrieve_content(self, file_id: str) -> Any:
        """Retrieve the contents of the specified file.
        
        Args:
            file_id: The ID of the file to retrieve content from
        
        Returns:
            The file content
        """
        content = self.client.files.content(file_id)
        return content

In [54]:
from openai import OpenAI
from typing import Optional, Dict, Any, List, Union


class VectorStoreClient:
    """Client for managing OpenAI Vector Stores.
    
    This client provides methods for:
    - Creating vector stores
    - Listing vector stores
    - Retrieving vector stores
    - Updating/modifying vector stores
    - Deleting vector stores
    - Searching within vector stores
    """
    
    def __init__(self, api_key: Optional[str] = None):
        """Initialize the VectorStoreClient.
        
        Args:
            api_key: Optional OpenAI API key. If not provided, will use OPENAI_API_KEY env variable.
        """
        self.client = OpenAI(api_key=api_key) if api_key else OpenAI()
    
    def create(
        self,
        name: Optional[str] = None,
        file_ids: Optional[List[str]] = None,
        chunking_strategy: Optional[Dict[str, Any]] = None,
        expires_after: Optional[Dict[str, Any]] = None,
        metadata: Optional[Dict[str, str]] = None
    ) -> Dict[str, Any]:
        """Create a new vector store.
        
        Args:
            name: The name of the vector store
            file_ids: List of File IDs that the vector store should use
            chunking_strategy: The chunking strategy used to chunk the file(s)
            expires_after: The expiration policy for the vector store
            metadata: Set of up to 16 key-value pairs (max 64 chars for keys, 512 for values)
        
        Returns:
            The created vector store object
        """
        kwargs = {}
        if name is not None:
            kwargs['name'] = name
        if file_ids is not None:
            kwargs['file_ids'] = file_ids
        if chunking_strategy is not None:
            kwargs['chunking_strategy'] = chunking_strategy
        if expires_after is not None:
            kwargs['expires_after'] = expires_after
        if metadata is not None:
            kwargs['metadata'] = metadata
        
        vector_store = self.client.vector_stores.create(**kwargs)
        return vector_store
    
    def list(
        self,
        limit: int = 20,
        order: str = "desc",
        after: Optional[str] = None,
        before: Optional[str] = None
    ) -> Dict[str, Any]:
        """List vector stores.
        
        Args:
            limit: Number of objects to return (1-100, default 20)
            order: Sort order by created_at timestamp ('asc' or 'desc', default 'desc')
            after: Cursor for pagination (object ID to start after)
            before: Cursor for pagination (object ID to start before)
        
        Returns:
            A list of vector store objects
        """
        kwargs = {'limit': limit, 'order': order}
        if after is not None:
            kwargs['after'] = after
        if before is not None:
            kwargs['before'] = before
        
        vector_stores = self.client.vector_stores.list(**kwargs)
        return vector_stores
    
    def retrieve(self, vector_store_id: str) -> Dict[str, Any]:
        """Retrieve a specific vector store by ID.
        
        Args:
            vector_store_id: The ID of the vector store to retrieve
        
        Returns:
            The vector store object
        """
        vector_store = self.client.vector_stores.retrieve(
            vector_store_id=vector_store_id
        )
        return vector_store
    
    def update(
        self,
        vector_store_id: str,
        name: Optional[str] = None,
        expires_after: Optional[Dict[str, Any]] = None,
        metadata: Optional[Dict[str, str]] = None
    ) -> Dict[str, Any]:
        """Update/modify a vector store.
        
        Args:
            vector_store_id: The ID of the vector store to modify
            name: The new name for the vector store
            expires_after: The expiration policy for the vector store
            metadata: Set of up to 16 key-value pairs
        
        Returns:
            The modified vector store object
        """
        kwargs = {'vector_store_id': vector_store_id}
        if name is not None:
            kwargs['name'] = name
        if expires_after is not None:
            kwargs['expires_after'] = expires_after
        if metadata is not None:
            kwargs['metadata'] = metadata
        
        vector_store = self.client.vector_stores.update(**kwargs)
        return vector_store
    
    def delete(self, vector_store_id: str) -> Dict[str, Any]:
        """Delete a vector store.
        
        Args:
            vector_store_id: The ID of the vector store to delete
        
        Returns:
            Deletion status object
        """
        deleted_vector_store = self.client.vector_stores.delete(
            vector_store_id=vector_store_id
        )
        return deleted_vector_store
    
    def search(
        self,
        vector_store_id: str,
        query: Union[str, List[str]],
        filters: Optional[Dict[str, Any]] = None,
        max_num_results: int = 10,
        ranking_options: Optional[Dict[str, Any]] = None,
        rewrite_query: bool = False
    ) -> Dict[str, Any]:
        """Search a vector store for relevant chunks.
        
        Args:
            vector_store_id: The ID of the vector store to search
            query: A query string or array for search
            filters: Filter to apply based on file attributes
            max_num_results: Maximum number of results (1-50, default 10)
            ranking_options: Ranking options for search
            rewrite_query: Whether to rewrite the natural language query for vector search
        
        Returns:
            A page of search results from the vector store
        """
        kwargs = {
            'vector_store_id': vector_store_id,
            'query': query,
            'max_num_results': max_num_results,
            'rewrite_query': rewrite_query
        }
        if filters is not None:
            kwargs['filters'] = filters
        if ranking_options is not None:
            kwargs['ranking_options'] = ranking_options
        
        # Note: The search endpoint might require direct API call if not in SDK yet
        search_results = self.client.vector_stores.search(**kwargs)
        return search_results
    
    def list_files(self, vector_store_id: str) -> Dict[str, Any]:
        """List files associated with a specific vector store.
        
        Args:
            vector_store_id: The ID of the vector store
        
        Returns:
            A list of files associated with the vector store
        """
        files = self.client.vector_stores.files.list(vector_store_id=vector_store_id)
        return files

In [18]:
class VectorStoreFileClient:
    """Client for managing files within OpenAI Vector Stores.
    
    This client provides methods for:
    - Creating vector store files (attaching files to vector stores)
    - Listing vector store files
    - Retrieving vector store files
    - Retrieving vector store file content
    - Updating vector store file attributes
    - Deleting vector store files
    """
    
    def __init__(self, api_key: Optional[str] = None):
        """Initialize the VectorStoreFileClient.
        
        Args:
            api_key: Optional OpenAI API key. If not provided, will use OPENAI_API_KEY env variable.
        """
        self.client = OpenAI(api_key=api_key) if api_key else OpenAI()
    
    def create(
        self,
        vector_store_id: str,
        file_id: str,
        attributes: Optional[Dict[str, Any]] = None,
        chunking_strategy: Optional[Dict[str, Any]] = None
    ) -> Dict[str, Any]:
        """Create a vector store file by attaching a File to a vector store.
        
        Args:
            vector_store_id: The ID of the vector store for which to create a File
            file_id: A File ID that the vector store should use
            attributes: Set of up to 16 key-value pairs (max 64 chars for keys, 512 for values/booleans/numbers)
            chunking_strategy: The chunking strategy used to chunk the file(s)
        
        Returns:
            The created vector store file object
        """
        kwargs = {
            'vector_store_id': vector_store_id,
            'file_id': file_id
        }
        if attributes is not None:
            kwargs['attributes'] = attributes
        if chunking_strategy is not None:
            kwargs['chunking_strategy'] = chunking_strategy
        
        vector_store_file = self.client.vector_stores.files.create(**kwargs)
        return vector_store_file
    
    def list(
        self,
        vector_store_id: str,
        limit: int = 20,
        order: str = "desc",
        after: Optional[str] = None,
        before: Optional[str] = None,
        filter: Optional[str] = None
    ) -> Dict[str, Any]:
        """List vector store files.
        
        Args:
            vector_store_id: The ID of the vector store that the files belong to
            limit: Number of objects to return (1-100, default 20)
            order: Sort order by created_at timestamp ('asc' or 'desc', default 'desc')
            after: Cursor for pagination (object ID to start after)
            before: Cursor for pagination (object ID to start before)
            filter: Filter by file status (in_progress, completed, failed, cancelled)
        
        Returns:
            A list of vector store file objects
        """
        kwargs = {
            'vector_store_id': vector_store_id,
            'limit': limit,
            'order': order
        }
        if after is not None:
            kwargs['after'] = after
        if before is not None:
            kwargs['before'] = before
        if filter is not None:
            kwargs['filter'] = filter
        
        vector_store_files = self.client.vector_stores.files.list(**kwargs)
        return vector_store_files
    
    def retrieve(
        self,
        vector_store_id: str,
        file_id: str
    ) -> Dict[str, Any]:
        """Retrieve a vector store file.
        
        Args:
            vector_store_id: The ID of the vector store that the file belongs to
            file_id: The ID of the file being retrieved
        
        Returns:
            The vector store file object
        """
        vector_store_file = self.client.vector_stores.files.retrieve(
            vector_store_id=vector_store_id,
            file_id=file_id
        )
        return vector_store_file
    
    def retrieve_content(
        self,
        vector_store_id: str,
        file_id: str
    ) -> Dict[str, Any]:
        """Retrieve the parsed contents of a vector store file.
        
        Args:
            vector_store_id: The ID of the vector store
            file_id: The ID of the file within the vector store
        
        Returns:
            The parsed contents of the specified vector store file
        """
        # Note: This endpoint returns the actual parsed content of the file
        # The SDK might use a different method name or require direct API call
        file_content = self.client.vector_stores.files.retrieve_content(
            vector_store_id=vector_store_id,
            file_id=file_id
        )
        return file_content
    
    def update_attributes(
        self,
        vector_store_id: str,
        file_id: str,
        attributes: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Update attributes on a vector store file.
        
        Args:
            vector_store_id: The ID of the vector store the file belongs to
            file_id: The ID of the file to update attributes
            attributes: Set of up to 16 key-value pairs (max 64 chars for keys, 512 for values/booleans/numbers)
        
        Returns:
            The updated vector store file object
        """
        updated_file = self.client.vector_stores.files.update(
            vector_store_id=vector_store_id,
            file_id=file_id,
            attributes=attributes
        )
        return updated_file
    
    def delete(
        self,
        vector_store_id: str,
        file_id: str
    ) -> Dict[str, Any]:
        """Delete a vector store file.
        
        Note: This removes the file from the vector store but the file itself is not deleted.
        To delete the file, use the delete file endpoint.
        
        Args:
            vector_store_id: The ID of the vector store that the file belongs to
            file_id: The ID of the file to delete
        
        Returns:
            Deletion status object
        """
        deleted_file = self.client.vector_stores.files.delete(
            vector_store_id=vector_store_id,
            file_id=file_id
        )
        return deleted_file

In [ ]:
from openai import OpenAI
import json

client = OpenAI(api_key=api_key)

In [55]:
user_name = "rahul"
user_id = "001"
vector_store_id  = 'vs_68ee383962c88191a77ec20f9e936b00'

vs_client = VectorStoreClient(api_key=api_key)


if not vs_client.retrieve(vector_store_id=vector_store_id):

    vector_store = vs_client.create(
        name=f"{user_name}_vs",
        metadata={"user_id": user_id},
        expires_after={'anchor':'last_active_at',"days": 1}
    )

else: 
    vector_store = vs_client.retrieve(vector_store_id=vector_store_id)

print(vector_store,)



VectorStore(id='vs_68ee383962c88191a77ec20f9e936b00', created_at=1760442425, file_counts=FileCounts(cancelled=0, completed=0, failed=1, in_progress=0, total=1), last_active_at=1760447006, metadata={'user_id': '001'}, name='rahul_vs', object='vector_store', status='completed', usage_bytes=0, expires_after=ExpiresAfter(anchor='last_active_at', days=1), expires_at=1760533406, description=None)


In [42]:
file_path = "/Users/rahulpatil/Downloads/Online Application Receipt.pdf"
file_client = FileClient(api_key=api_key)
file_upload = file_client.create(
    file=open(file_path, "rb"),
    purpose="user_data",
    expires_after={'anchor':'created_at',"seconds": 86400}
)

    

In [81]:
file_client.retrieve(file_id=file_upload.id)

FileObject(id='file-12AFQroB7QNJ8SfTKKzLHU', bytes=3292032, created_at=1760443108, filename='Online Application Receipt.pdf', object='file', purpose='user_data', status='processed', expires_at=1760529508, status_details=None)

In [82]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

response = client.responses.create(
    model="gpt-4.1",
    input=[
        {
            "role": "user",
            "content": [
                { "type": "input_text", "text": "what is my passport application date" },
                {
                    "type": "input_file",
                    "file_id": "file-12AFQroB7QNJ8SfTKKzLHU"
                }
                ]
        }
    ]
)

print(response)

Response(id='resp_0fd2bcfd978918730068ee4ceb9b608195a6ea21acf973bda5', created_at=1760447725.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_0fd2bcfd978918730068ee4cef21308195ac0f400c2664051e', content=[ResponseOutputText(annotations=[], text='Your passport application date, as shown in the document, is **24/06/2023** (24th June 2023). \n\nYou can find it in the "Date of Application" field in the "Applicants Details" section of your Online Application Receipt.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, reasoning=Reasoning(effort=None, generate_summary=None, summary=None), safety_identifier=None, service_t

In [229]:
client.vector_stores.list()

SyncCursorPage[VectorStore](data=[VectorStore(id='vs_68efa3e91ed08191a60bc7096e31cc03', created_at=1760535529, file_counts=FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1), last_active_at=1760535530, metadata={'user_id': 'a346cd42-b4f6-4408-b34f-eea92d5eb66a'}, name='Rahul S Patil_knowledge_base', object='vector_store', status='completed', usage_bytes=2081, expires_after=ExpiresAfter(anchor='last_active_at', days=365), expires_at=1792071530, description=None)], has_more=False, object='list', first_id='vs_68efa3e91ed08191a60bc7096e31cc03', last_id='vs_68efa3e91ed08191a60bc7096e31cc03')

In [230]:
client.vector_stores.files.list(vector_store_id='vs_68efa3e91ed08191a60bc7096e31cc03')

SyncCursorPage[VectorStoreFile](data=[VectorStoreFile(id='file-W8XBjMgA9GmBHzWRgCodmm', created_at=1760535530, last_error=None, object='vector_store.file', status='completed', usage_bytes=2081, vector_store_id='vs_68efa3e91ed08191a60bc7096e31cc03', attributes={'knowledge_item_id': '432840f4-103d-452c-8e4f-a4a45abbc4ce', 'folder_id': '8c888946-12b5-4e30-b439-8f1e2b63eda8', 'title': 'textaenderungen_ILD_new', 'source_type': 'upload'}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))], has_more=False, object='list', first_id='file-W8XBjMgA9GmBHzWRgCodmm', last_id='file-W8XBjMgA9GmBHzWRgCodmm')

In [223]:
client.vector_stores.delete(vector_store_id='vs_68ee383962c88191a77ec20f9e936b00')

VectorStoreDeleted(id='vs_68ee383962c88191a77ec20f9e936b00', deleted=True, object='vector_store.deleted')

In [125]:


response = client.vector_stores.files.upload_and_poll(        # Upload file
    vector_store_id=vector_store.id,
    file=open("/Users/rahulpatil/Documents/workspace/synapse/scripts/passport_test_document.txt", "rb")
)

In [126]:
response

VectorStoreFile(id='file-VVYBebv3Mn4kpZLBC9KpxS', created_at=1760448746, last_error=None, object='vector_store.file', status='completed', usage_bytes=12882, vector_store_id='vs_68ee383962c88191a77ec20f9e936b00', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [148]:


results

SyncPage[VectorStoreSearchResponse](data=[VectorStoreSearchResponse(attributes={}, content=[Content(text="pplicants Details:\nApplication Reference No.\n(ARN)\nService Type\nType of Application\nGiven Name\nSurname\nGender\nFather's Name\nMother's Name\nSpouse's Name\nDate Of Birth\nPlace of Birth\nMarital Status\nEmployment Type\nPresent Residential Address\nFile Number\nSlot Type\n25-0067474245\nREISSUE\nNormal\nPRATIK ANIL\nDEO\nMALE\nANIL DAGADU DEO\nSUNITA ANIL DEO\nSHIVANI SUBHASH HANDE\n10/04/1995\nJALGAON, JALGAON, MAHARASHTRA\nMARRIED\nPRIVATE\n14, A3, SEEMA GARDEN, SHIKSHAK\nNAGAR, PAUD ROAD, KOTHRUD, Pune\nCity, 411038, Maharashtra, INDIA\nPayment Details:#\nTotal Fee (Rs.)\nPaid Fee (Rs.)\nDate and Time\nTransaction Id\nAppointment Details:\nPassport Seva Kendra\nAddress\nAppointment Id\nAppointment Date and\nTime\nReporting Date and Time\nBatch 21\nSequence No 24\n2000.00 2000.00\n04/10/2025 05:33 PM\nCPAFSICCD2\nZero One, S.No. 79/1, Ghorpadi - Mundhwa\nRoad, Pingle Wasti

In [154]:
results.data

[VectorStoreSearchResponse(attributes={}, content=[Content(text="pplicants Details:\nApplication Reference No.\n(ARN)\nService Type\nType of Application\nGiven Name\nSurname\nGender\nFather's Name\nMother's Name\nSpouse's Name\nDate Of Birth\nPlace of Birth\nMarital Status\nEmployment Type\nPresent Residential Address\nFile Number\nSlot Type\n25-0067474245\nREISSUE\nNormal\nPRATIK ANIL\nDEO\nMALE\nANIL DAGADU DEO\nSUNITA ANIL DEO\nSHIVANI SUBHASH HANDE\n10/04/1995\nJALGAON, JALGAON, MAHARASHTRA\nMARRIED\nPRIVATE\n14, A3, SEEMA GARDEN, SHIKSHAK\nNAGAR, PAUD ROAD, KOTHRUD, Pune\nCity, 411038, Maharashtra, INDIA\nPayment Details:#\nTotal Fee (Rs.)\nPaid Fee (Rs.)\nDate and Time\nTransaction Id\nAppointment Details:\nPassport Seva Kendra\nAddress\nAppointment Id\nAppointment Date and\nTime\nReporting Date and Time\nBatch 21\nSequence No 24\n2000.00 2000.00\n04/10/2025 05:33 PM\nCPAFSICCD2\nZero One, S.No. 79/1, Ghorpadi - Mundhwa\nRoad, Pingle Wasti, Opp. Ganga\nOrchid, Pune\n1000775683438

In [218]:
developer_prompt = """
  You are an intelligent knowledge retrieval assistant with access to multiple information sources.

  ## Query Classification & Response Strategy:

  1. **Knowledge Base Queries**: If the query relates to information that could be in the user's knowledge base (documents, personal data, organizational content), FIRST search the provided sources.

  2. **Real-time Information**: If the query requires current data (weather, news, stock prices, today's events), use web search tools.

  3. **Hybrid Queries**: If the query combines personal context with creative/general tasks:
     - First, search the knowledge base for relevant personal information
     - Then, use that context combined with general knowledge to generate a personalized response

  4. **General Knowledge**: If the query is educational or creative without personal context, use your general knowledge directly.

  ## Contextual Intelligence:
  - **Apply knowledge to retrieved data**: When you have specific information from the knowledge base and the user asks about it, analyze THAT specific data using your general knowledge
  - **Recognize follow-up questions**: References like "my", "those", "this", "that" indicate the user wants you to explain or work with previously retrieved information
  - **Don't just describe - analyze**: If the user asks about specific data you've already retrieved, provide insights about THAT data, not just generic explanations of what such data could mean
  - **Use the actual values**: When explaining concepts related to retrieved data, reference and work with the actual values, formats, and patterns present in the user's specific information

  ## Response Guidelines:
  - Be precise and concise in your answers
  - Clearly cite sources when using knowledge base information
  - When combining KB data with general knowledge, explicitly show how the general knowledge applies to their specific data
  - **Always ground your explanations in the actual data retrieved** - don't give abstract possibilities when you have concrete information
  - State your reasoning when making inferences (e.g., "Looking at your specific [data], this indicates...")
  - If information is not available in expected sources, explain this and use the most appropriate alternative source

  ## Priority Order:
  Knowledge Base (if relevant) → Apply General Knowledge to Retrieved Data → Web Search (if real-time) → General Knowledge
"""

temp = None
user_input = input("Enter your query: ")
previous_response_id = None
while user_input.lower() != "exit":
    response = client.responses.create(
        model="gpt-4.1",
        tools=[{
          "type": "file_search",
          "vector_store_ids": [vector_store.id],
          "max_num_results": 20
        },
        {
          "type": "web_search_preview"
        }],
        instructions=developer_prompt,
        input=user_input,
        previous_response_id = previous_response_id
    )
    print("---"*50)
    print("User Input:", user_input)
    response_output_text = [item for item in response.output if isinstance(item, ResponseOutputMessage)][0].content[0].text
    print("Response:", response)
    previous_response_id = response.id
    temp = response
    print("Response:", response.output)
    user_input = input("Enter your query (or type 'exit' to quit): ")

------------------------------------------------------------------------------------------------------------------------------------------------------
User Input: what is gil in python
Response: The Global Interpreter Lock (GIL) in Python is a mutex (mutual exclusion lock) that protects access to Python objects, preventing multiple native threads from executing Python bytecodes at once. This means that even in multi-threaded programs, only one thread can execute Python code at a time per process. The GIL is necessary because CPython's memory management is not thread-safe.

- This design simplifies the implementation of CPython, especially with regard to memory management.
- However, it also means that multi-threaded Python programs may not see a performance gain on multi-core systems for CPU-bound operations.
- It does not affect I/O-bound programs as much, since threads waiting for I/O release the GIL, allowing other threads to run.

In summary, the GIL makes Python threading less eff

AttributeError: 'str' object has no attribute 'id'

In [217]:
from openai.types.responses.response_output_message import ResponseOutputMessage
from openai.types.vector_stores.


True

In [206]:
response.output[0].content

[ResponseOutputText(annotations=[AnnotationFileCitation(file_id='file-VVYBebv3Mn4kpZLBC9KpxS', filename='passport_test_document.txt', index=66, type='file_citation')], text="Your father's name, according to your document, is Anil Dagadu Deo.", type='output_text', logprobs=[])]

In [118]:
client.vector_stores.files.list(vector_store_id=vector_store.id)

SyncCursorPage[VectorStoreFile](data=[VectorStoreFile(id='file-12AFQroB7QNJ8SfTKKzLHU', created_at=1760447006, last_error=LastError(code='invalid_file', message='The file could not be parsed because it is empty.'), object='vector_store.file', status='failed', usage_bytes=0, vector_store_id='vs_68ee383962c88191a77ec20f9e936b00', attributes={'source': 'admission_form', 'uploaded_by': 'rahul'}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))], has_more=False, object='list', first_id='file-12AFQroB7QNJ8SfTKKzLHU', last_id='file-12AFQroB7QNJ8SfTKKzLHU')

In [117]:
for i in response:
    print(i)

VectorStoreFile(id='file-12AFQroB7QNJ8SfTKKzLHU', created_at=1760447006, last_error=LastError(code='invalid_file', message='The file could not be parsed because it is empty.'), object='vector_store.file', status='failed', usage_bytes=0, vector_store_id='vs_68ee383962c88191a77ec20f9e936b00', attributes={'source': 'admission_form', 'uploaded_by': 'rahul'}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))


In [86]:
client.vector_stores.files.create(
    vector_store_id=vector_store.id,
    file_id=file_upload.id)

VectorStoreFile(id='file-12AFQroB7QNJ8SfTKKzLHU', created_at=1760447916, last_error=None, object='vector_store.file', status='in_progress', usage_bytes=0, vector_store_id='vs_68ee383962c88191a77ec20f9e936b00', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))

In [73]:
vector_store_file.

SyntaxError: invalid syntax (1703522779.py, line 1)